In [22]:
import os
import re
import json
import pandas as pd
from io import StringIO
from dotenv import load_dotenv
load_dotenv()
from scholarlm.utils import get_filenames_in_directory, table_extract, process_pdf

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
main_directory = os.getenv("POND_PATH")
pdf_directory = os.getenv("POND_PDF_PATH")
text_directory = os.getenv("POND_TEXT_PATH2")

with open(os.path.join(main_directory, "directory.json"), "r") as f:
    paper_info = json.load(f)

text_files = get_filenames_in_directory(text_directory, ignore = [".DS_Store"])
text_files.sort()

text_files = [
    'physical_and_chemical_limnological.txt',
    'physical-chemical_influences.txt',
    'prairie_wetland.txt',
    'net_heterotrophy.txt',
    'habitat_characteristics.txt',
    'biodiversity_of_constructed.txt',
    'fish_production_in_lakes.txt',
    'long-term_stability.txt',
    'diversity_of_macroinvertebrates.txt',
    'impact_of_macrophytes.txt'
]


text_filepaths = []
text_info = []
for f in text_files:
    paper_code = f.replace('.txt', '')
    filepath = os.path.join(text_directory, f)
    metadata = paper_info.get(paper_code, {})
    # ID Addition:
    metadata['paper_code'] = paper_code
    text_filepaths.append(filepath)
    text_info.append(metadata)

In [64]:
with open(text_filepaths[4], "r") as f:
    context = f.read()

tables = pd.read_html(StringIO(context))

In [65]:
print(context)

<page number="0">

See discussions, stats, and author profiles for this publication at: https://www.researchgate.net/publication/229087071

Habitat characteristics and odonate diversity in mountain ponds of central Italy

Article in Aquatic Conservation Marine and Freshwater Ecosystems · November 2005
DOI: 10.1002/aqc.741

CITATIONS
22

READS
97

3 authors, including:

G. Carchini
University of Rome Tor Vergata
51 PUBLICATIONS  514 CITATIONS
SEE PROFILE

Angelo G Solimini
Sapienza University of Rome
95 PUBLICATIONS  1,665 CITATIONS
SEE PROFILE

</page>

<page number="1">

THE ATRIUM, SOUTHERN GATE, CHICHESTER, WEST SUSSEX P019 8SQ

***IMMEDIATE RESPONSE REQUIRED***

Your article may be published online via Wiley's EarlyView® service (www.interscience.wiley.com) shortly after receipt of corrections. EarlyView® is Wiley's online publication of individual articles in full-text HTML and/or pdf format before release of the compiled print issue of the journal. Articles posted online in Early

In [66]:
tables[0]

,index,Altitude (m),Surface area (ha),Maximum depth (m),Chl-a (µg L-1),NO3- (mg L-1),NH4+ (mg L-1),PO43- (mg L-1),Conductivity (µS cm-1),pH,...,Cercion lindeni,Enallagma cyathigerum,Ischnura elegans,Aeshna cyanea,Anax imperator,Libellula depressa,Orthetrum coerulescens,Crocothemis erythraea,Sympetrum flaveolum,Sympetrum fonscolombii
0,AQG,1164,1.15,0.90,271,5.89,0.24,0.11,99,9.7,...,0,0,1,0,0,0,0,0,0,0
1,AQP,1164,0.13,1.00,41,3.52,1.02,0.10,278,8.1,...,0,0,0,0,0,0,0,0,0,0
2,BAG,1780,0.07,0.50,1479,2.70,6.46,0.31,464,9.6,...,0,0,0,0,0,0,0,0,0,0
3,BAR,1604,1.52,2.05,143,6.49,0.19,0.05,161,9.1,...,0,1,0,0,0,0,0,0,0,0
4,CAG,2000,0.10,0.90,1704,2.33,3.04,0.06,275,8.1,...,0,0,0,0,0,0,0,0,0,0
5,CAL,1115,0.47,3.50,123,4.07,0.10,0.05,207,9.1,...,0,0,0,0,1,0,0,0,0,0
6,CAM,1528,0.26,0.40,2,3.58,0.12,0.07,228,8.3,...,0,0,0,0,0,1,0,0,1,0
7,CAP,2005,0.04,1.60,446,2.83,0.40,0.07,143,8.8,...,0,0,0,0,0,0,0,0,0,0
8,COR,1261,1.04,1.00,1137,4.02,0.41,0.10,201,8.7,...,0,0,0,0,0,0,0,0,0,0
9,DUC,1788,3.50,1.65,55,2.92,0.61,0.82,234,8.9,...,0,1,0,0,0,1,0,0,0,0


In [62]:
identification_prompt = (
    "You are an expert in identifying ponds, lakes, and wetlands referenced in text from scientific literature. "
    "Using the given context, find, identify, and list all individual pond, lake, or wetland ecosystems it mentions. "
    "Any identified ecosystem must be a distinct entity, and not a general reference to or an aggregate collection of ecosystems. "
    "For each ecosystem, provide the following identification attributes: name, date observed, geographic location, and ecosystem type (pond, lake, wetland, or other). "
    "If any one of these attributes is not explicitly mentioned in the text, respond with the value None for that attribute. "
    "Each identified entity should be as specific as possible to the state it was measured in. "
    "If there are multiple dates observed for what is otherwise the same ecosystem, treat each as separate, identified items. "
    "If an ecosystem is measured in different physical states (e.g. after treatment or during different seasons), treat each as separate, identified items, and distinguish them in the name attribute. "
    "However, if any ecosystem is mentioned multiple times with identical attributes or states, only list it once. "
    "If the text uses multiple names that refer to the same ecosystem, use only the most complete, full-form name that still uniquely identifies the ecosystem. "
    "It is also acceptable to use codes or abbreviations if that is the only form of the name given. "
    "Along with the body of the text, it is also acceptable to use ecosystems identified within tables. "
    "Format the output as a JSON object with an array of items, where each item is an object "
    "containing the specified identification attributes. For example:\n"
    "{{'items': [{{'name': 'Pond A', 'date': '2020-05-01', 'location': 'Location A', 'ecosystem': 'pond'}}, {{'name': 'Pond A', 'date': '2021-05-01', 'location': 'Location A', 'ecosystem': 'pond'}}, {{'name': 'Wetland B', 'date': None, 'location': None, 'ecosystem': 'wetland'}}]}}\n"
    "If no distinct ecosystems are found, respond with an empty list. For example: "
    "{{'items': []}}"
)

In [63]:
print(identification_prompt)

You are an expert in identifying ponds, lakes, and wetlands referenced in text from scientific literature. Using the given context, find, identify, and list all individual pond, lake, or wetland ecosystems it mentions. Any identified ecosystem must be a distinct entity, and not a general reference to or an aggregate collection of ecosystems. For each ecosystem, provide the following identification attributes: name, date observed, geographic location, and ecosystem type (pond, lake, wetland, or other). If any one of these attributes is not explicitly mentioned in the text, respond with the value None for that attribute. Each identified entity should be as specific as possible to the state it was measured in. If there are multiple dates observed for what is otherwise the same ecosystem, treat each as separate, identified items. If an ecosystem is measured in different physical states (e.g. after treatment or during different seasons), treat each as separate, identified items, and disting